# 17. Observer, command, template, registry, and composite

**Level:** Intermediate to advanced

**Learning outcome:** Represent events, reversible local commands, extension hooks, and recursive groups explicitly.

**Prerequisites:** 04-07 and 16: collaboration patterns and failure behavior.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](16_Strategy_Factory_Adapter_Facade_and_Decorator.ipynb) | [Next notebook](18_Desired_State_Idempotency_and_Rollback.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Intermediate to advanced

**Learning goal:** Use small coordinated objects to express workflows and extension points.

**Observer** delivers an event to registered listeners. An in-memory teaching bus does not
provide durable delivery, retries, ordering across processes, or isolation from slow listeners.
Decide what happens when one listener fails; this notebook's simple bus propagates failures.

**Command objects** represent an action as data plus behavior. This helps review, queuing,
audit, and simulation. An undo action is not automatically safe on real devices: state may
have changed since execution, and external side effects may be irreversible.

**Template method** keeps an overall algorithm in a base class while subclasses supply a
designated step. Use it when that fixed sequence is genuinely shared. If every subclass
needs to alter every step, an injected pipeline is clearer.

**Registries** map stable names to known implementations. Enforce uniqueness and make
registration explicit unless the application genuinely benefits from class-creation hooks.
Unknown entries and duplicates are errors, not cues to load arbitrary external code.

**Composite** lets a group of objects and an individual object expose the same operation.
A site group can sum device counts without every caller special-casing nesting. Real
hierarchies must prevent cycles or define how repeated references are handled.

Keep these implementations small enough that the mechanism is visible. The exercises ask you
to reason about ownership, failure behavior, and idempotency instead of memorizing pattern names.

**Pause and explain:** Explain why an in-memory observer is not a durable queue and why undo is not guaranteed rollback.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 056: Observer notifies registered listeners

**Core concept:** Publish an event through explicit listener registration.

**Network-engineering purpose:** A simulated interface event can update an audit collector without hard-coding it into the publisher.

### Read the implementation in this order
1. Store listeners in instance state.
2. Subscribe a callable.
3. Publish to a snapshot of the listener list.
4. Check that the listener received exactly the event sent.

**Common mistake / boundary:** This synchronous bus propagates listener exceptions and has no persistence.

**Practice after the ten worked scenarios:** Add unsubscribe and decide how duplicate subscriptions should behave.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0551 - Observer notifies registered listeners: branch scenario

**Scenario 01/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |
| `healthy` | `True` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet1/1', 'up': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [1]:
# Example 0551: Observer notifies registered listeners - branch
case = {'interface': 'Ethernet1/1', 'healthy': True}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet1/1', 'up': True}]


### Example 0552 - Observer notifies registered listeners: campus scenario

**Scenario 02/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |
| `healthy` | `False` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet2/1', 'up': False}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [2]:
# Example 0552: Observer notifies registered listeners - campus
case = {'interface': 'Ethernet2/1', 'healthy': False}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet2/1', 'up': False}]


### Example 0553 - Observer notifies registered listeners: dc scenario

**Scenario 03/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |
| `healthy` | `True` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet3/1', 'up': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [3]:
# Example 0553: Observer notifies registered listeners - dc
case = {'interface': 'Ethernet3/1', 'healthy': True}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet3/1', 'up': True}]


### Example 0554 - Observer notifies registered listeners: wan scenario

**Scenario 04/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |
| `healthy` | `True` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet4/1', 'up': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [4]:
# Example 0554: Observer notifies registered listeners - wan
case = {'interface': 'Ethernet4/1', 'healthy': True}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet4/1', 'up': True}]


### Example 0555 - Observer notifies registered listeners: lab scenario

**Scenario 05/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |
| `healthy` | `False` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet5/1', 'up': False}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [5]:
# Example 0555: Observer notifies registered listeners - lab
case = {'interface': 'Ethernet5/1', 'healthy': False}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet5/1', 'up': False}]


### Example 0556 - Observer notifies registered listeners: wireless scenario

**Scenario 06/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |
| `healthy` | `True` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet6/1', 'up': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [6]:
# Example 0556: Observer notifies registered listeners - wireless
case = {'interface': 'Ethernet6/1', 'healthy': True}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet6/1', 'up': True}]


### Example 0557 - Observer notifies registered listeners: remote scenario

**Scenario 07/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |
| `healthy` | `False` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet7/1', 'up': False}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [7]:
# Example 0557: Observer notifies registered listeners - remote
case = {'interface': 'Ethernet7/1', 'healthy': False}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet7/1', 'up': False}]


### Example 0558 - Observer notifies registered listeners: core scenario

**Scenario 08/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |
| `healthy` | `True` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet8/1', 'up': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [8]:
# Example 0558: Observer notifies registered listeners - core
case = {'interface': 'Ethernet8/1', 'healthy': True}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet8/1', 'up': True}]


### Example 0559 - Observer notifies registered listeners: mgmt scenario

**Scenario 09/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |
| `healthy` | `False` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet9/1', 'up': False}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [9]:
# Example 0559: Observer notifies registered listeners - mgmt
case = {'interface': 'Ethernet9/1', 'healthy': False}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet9/1', 'up': False}]


### Example 0560 - Observer notifies registered listeners: backup scenario

**Scenario 10/10; pattern 056; Intermediate to advanced.** This is a self-contained offline exercise. A simulated interface event can update an audit collector without hard-coding it into the publisher.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |
| `healthy` | `True` |

**Logic to trace:** Store listeners in instance state. Check that the listener received exactly the event sent. Follow the class-specific steps under Pattern 056 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'interface': 'Ethernet10/1', 'up': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This synchronous bus propagates listener exceptions and has no persistence.

In [10]:
# Example 0560: Observer notifies registered listeners - backup
case = {'interface': 'Ethernet10/1', 'healthy': True}

class EventBus:
    def __init__(self):
        self.listeners = []

    def subscribe(self, listener):
        self.listeners.append(listener)

    def publish(self, event):
        for listener in tuple(self.listeners):
            listener(event)

events = []
bus = EventBus()
bus.subscribe(events.append)
event = {"interface": case["interface"], "up": case["healthy"]}
bus.publish(event)
assert events == [event]
print(events)


[{'interface': 'Ethernet10/1', 'up': True}]


## Pattern 057: Command object with simulated undo

**Core concept:** An action can retain enough prior state for a local reversal.

**Network-engineering purpose:** A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

### Read the implementation in this order
1. Keep the receiver object separate from the command.
2. Record the old value at execution.
3. Set the proposed value.
4. Undo and verify the original state returns.

**Common mistake / boundary:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

**Practice after the ten worked scenarios:** Reject undo before execute and reject a second execute until the first is resolved.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0561 - Command object with simulated undo: branch scenario

**Scenario 01/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [11]:
# Example 0561: Command object with simulated undo - branch
case = {'mtu': 1500}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0562 - Command object with simulated undo: campus scenario

**Scenario 02/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [12]:
# Example 0562: Command object with simulated undo - campus
case = {'mtu': 9000}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0563 - Command object with simulated undo: dc scenario

**Scenario 03/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [13]:
# Example 0563: Command object with simulated undo - dc
case = {'mtu': 9216}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0564 - Command object with simulated undo: wan scenario

**Scenario 04/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [14]:
# Example 0564: Command object with simulated undo - wan
case = {'mtu': 576}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0565 - Command object with simulated undo: lab scenario

**Scenario 05/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [15]:
# Example 0565: Command object with simulated undo - lab
case = {'mtu': 1600}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0566 - Command object with simulated undo: wireless scenario

**Scenario 06/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [16]:
# Example 0566: Command object with simulated undo - wireless
case = {'mtu': 1500}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0567 - Command object with simulated undo: remote scenario

**Scenario 07/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [17]:
# Example 0567: Command object with simulated undo - remote
case = {'mtu': 2000}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0568 - Command object with simulated undo: core scenario

**Scenario 08/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [18]:
# Example 0568: Command object with simulated undo - core
case = {'mtu': 4470}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0569 - Command object with simulated undo: mgmt scenario

**Scenario 09/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [19]:
# Example 0569: Command object with simulated undo - mgmt
case = {'mtu': 9100}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


### Example 0570 - Command object with simulated undo: backup scenario

**Scenario 10/10; pattern 057; Intermediate to advanced.** This is a self-contained offline exercise. A reviewable desired-MTU command can demonstrate execution and rollback on an in-memory model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Keep the receiver object separate from the command. Undo and verify the original state returns. Follow the class-specific steps under Pattern 057 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
restored MTU: 1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Real-device undo needs concurrency and drift checks; this only reverses local assignment.

In [20]:
# Example 0570: Command object with simulated undo - backup
case = {'mtu': 8192}

from dataclasses import dataclass

@dataclass
class Interface:
    mtu: int

class SetMtu:
    def __init__(self, interface, proposed):
        self.interface, self.proposed, self.previous = interface, proposed, None

    def execute(self):
        self.previous = self.interface.mtu
        self.interface.mtu = self.proposed

    def undo(self):
        if self.previous is None:
            raise RuntimeError("command not executed")
        self.interface.mtu = self.previous

interface = Interface(1500)
command = SetMtu(interface, case["mtu"])
command.execute()
assert interface.mtu == case["mtu"]
command.undo()
assert interface.mtu == 1500
print("restored MTU:", interface.mtu)


restored MTU: 1500


## Pattern 058: Template method fixes workflow order

**Core concept:** A base workflow can define sequencing and delegate one transformation.

**Network-engineering purpose:** A parser pipeline can always strip whitespace before platform-specific normalization.

### Read the implementation in this order
1. Define run in the base.
2. Delegate normalize to a subclass hook.
3. Override only the hook.
4. Check the public run method performs both steps in order.

**Common mistake / boundary:** A hook returning the wrong type can violate the base algorithm's contract.

**Practice after the ten worked scenarios:** Add a second subclass that preserves case and compare outputs.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0571 - Template method fixes workflow order: branch scenario

**Scenario 01/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
branch-edge-01
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [21]:
# Example 0571: Template method fixes workflow order - branch
case = {'hostname': 'branch-edge-01'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


branch-edge-01


### Example 0572 - Template method fixes workflow order: campus scenario

**Scenario 02/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
campus-access-02
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [22]:
# Example 0572: Template method fixes workflow order - campus
case = {'hostname': 'campus-access-02'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


campus-access-02


### Example 0573 - Template method fixes workflow order: dc scenario

**Scenario 03/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
dc-leaf-03
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [23]:
# Example 0573: Template method fixes workflow order - dc
case = {'hostname': 'dc-leaf-03'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


dc-leaf-03


### Example 0574 - Template method fixes workflow order: wan scenario

**Scenario 04/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
wan-border-04
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [24]:
# Example 0574: Template method fixes workflow order - wan
case = {'hostname': 'wan-border-04'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


wan-border-04


### Example 0575 - Template method fixes workflow order: lab scenario

**Scenario 05/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
lab-router-05
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [25]:
# Example 0575: Template method fixes workflow order - lab
case = {'hostname': 'lab-router-05'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


lab-router-05


### Example 0576 - Template method fixes workflow order: wireless scenario

**Scenario 06/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
wireless-dist-06
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [26]:
# Example 0576: Template method fixes workflow order - wireless
case = {'hostname': 'wireless-dist-06'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


wireless-dist-06

### Example 0577 - Template method fixes workflow order: remote scenario

**Scenario 07/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
remote-gateway-07
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [27]:
# Example 0577: Template method fixes workflow order - remote
case = {'hostname': 'remote-gateway-07'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


remote-gateway-07


### Example 0578 - Template method fixes workflow order: core scenario

**Scenario 08/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
core-router-08
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [28]:
# Example 0578: Template method fixes workflow order - core
case = {'hostname': 'core-router-08'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


core-router-08


### Example 0579 - Template method fixes workflow order: mgmt scenario

**Scenario 09/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
mgmt-switch-09
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [29]:
# Example 0579: Template method fixes workflow order - mgmt
case = {'hostname': 'mgmt-switch-09'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


mgmt-switch-09


### Example 0580 - Template method fixes workflow order: backup scenario

**Scenario 10/10; pattern 058; Intermediate to advanced.** This is a self-contained offline exercise. A parser pipeline can always strip whitespace before platform-specific normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define run in the base. Check the public run method performs both steps in order. Follow the class-specific steps under Pattern 058 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
backup-edge-10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A hook returning the wrong type can violate the base algorithm's contract.

In [30]:
# Example 0580: Template method fixes workflow order - backup
case = {'hostname': 'backup-edge-10'}

from abc import ABC, abstractmethod

class Pipeline(ABC):
    def run(self, text):
        return self.normalize(text.strip())

    @abstractmethod
    def normalize(self, text):
        raise NotImplementedError

class LowercasePipeline(Pipeline):
    def normalize(self, text):
        return text.lower()

result = LowercasePipeline().run("  " + case["hostname"].upper() + "  ")
assert result == case["hostname"]
print(result)


backup-edge-10

## Pattern 059: Explicit registry rejects duplicates

**Core concept:** An application-controlled map can provide a predictable extension point.

**Network-engineering purpose:** Known checks can be looked up by name without importing untrusted modules.

### Read the implementation in this order
1. Give each registry its own mapping.
2. Reject a duplicate before storing it.
3. Resolve the registered callable.
4. Check duplicate failure and known lookup.

**Common mistake / boundary:** A module-global mutable registry can leak state between tests and notebook reruns.

**Practice after the ten worked scenarios:** Raise a domain-specific error for an unknown check name.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0581 - Explicit registry rejects duplicates: branch scenario

**Scenario 01/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [31]:
# Example 0581: Explicit registry rejects duplicates - branch
case = {'mtu': 1500}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected


### Example 0582 - Explicit registry rejects duplicates: campus scenario

**Scenario 02/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [32]:
# Example 0582: Explicit registry rejects duplicates - campus
case = {'mtu': 9000}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected

### Example 0583 - Explicit registry rejects duplicates: dc scenario

**Scenario 03/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [33]:
# Example 0583: Explicit registry rejects duplicates - dc
case = {'mtu': 9216}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected


### Example 0584 - Explicit registry rejects duplicates: wan scenario

**Scenario 04/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [34]:
# Example 0584: Explicit registry rejects duplicates - wan
case = {'mtu': 576}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected


### Example 0585 - Explicit registry rejects duplicates: lab scenario

**Scenario 05/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [35]:
# Example 0585: Explicit registry rejects duplicates - lab
case = {'mtu': 1600}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected

### Example 0586 - Explicit registry rejects duplicates: wireless scenario

**Scenario 06/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [36]:
# Example 0586: Explicit registry rejects duplicates - wireless
case = {'mtu': 1500}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected


### Example 0587 - Explicit registry rejects duplicates: remote scenario

**Scenario 07/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [37]:
# Example 0587: Explicit registry rejects duplicates - remote
case = {'mtu': 2000}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected

### Example 0588 - Explicit registry rejects duplicates: core scenario

**Scenario 08/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [38]:
# Example 0588: Explicit registry rejects duplicates - core
case = {'mtu': 4470}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected

### Example 0589 - Explicit registry rejects duplicates: mgmt scenario

**Scenario 09/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [39]:
# Example 0589: Explicit registry rejects duplicates - mgmt
case = {'mtu': 9100}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected

### Example 0590 - Explicit registry rejects duplicates: backup scenario

**Scenario 10/10; pattern 059; Intermediate to advanced.** This is a self-contained offline exercise. Known checks can be looked up by name without importing untrusted modules.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Give each registry its own mapping. Check duplicate failure and known lookup. Follow the class-specific steps under Pattern 059 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
duplicate registration rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A module-global mutable registry can leak state between tests and notebook reruns.

In [40]:
# Example 0590: Explicit registry rejects duplicates - backup
case = {'mtu': 8192}

class CheckRegistry:
    def __init__(self):
        self._checks = {}

    def register(self, name, check):
        if name in self._checks:
            raise ValueError(f"duplicate check: {name}")
        self._checks[name] = check

    def run(self, name, value):
        return self._checks[name](value)

registry = CheckRegistry()
registry.register("mtu", lambda value: 576 <= value <= 9216)
assert registry.run("mtu", case["mtu"])
try:
    registry.register("mtu", lambda value: True)
except ValueError:
    print("duplicate registration rejected")
else:
    raise AssertionError("duplicate overwritten")


duplicate registration rejected

## Pattern 060: Composite inventory groups

**Core concept:** A leaf and a group share the same counting operation.

**Network-engineering purpose:** Nested site inventory can be summarized without flattening every object manually.

### Read the implementation in this order
1. Give a leaf count() returning one.
2. Give a group count() summing child counts.
3. Nest one group within another.
4. Check the recursive result.

**Common mistake / boundary:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

**Practice after the ten worked scenarios:** Add cycle prevention or explicitly document a DAG counting policy.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0591 - Composite inventory groups: branch scenario

**Scenario 01/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [41]:
# Example 0591: Composite inventory groups - branch
case = {'hostname': 'branch-edge-01'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0592 - Composite inventory groups: campus scenario

**Scenario 02/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [42]:
# Example 0592: Composite inventory groups - campus
case = {'hostname': 'campus-access-02'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0593 - Composite inventory groups: dc scenario

**Scenario 03/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [43]:
# Example 0593: Composite inventory groups - dc
case = {'hostname': 'dc-leaf-03'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0594 - Composite inventory groups: wan scenario

**Scenario 04/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [44]:
# Example 0594: Composite inventory groups - wan
case = {'hostname': 'wan-border-04'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0595 - Composite inventory groups: lab scenario

**Scenario 05/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [45]:
# Example 0595: Composite inventory groups - lab
case = {'hostname': 'lab-router-05'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0596 - Composite inventory groups: wireless scenario

**Scenario 06/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [46]:
# Example 0596: Composite inventory groups - wireless
case = {'hostname': 'wireless-dist-06'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0597 - Composite inventory groups: remote scenario

**Scenario 07/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [47]:
# Example 0597: Composite inventory groups - remote
case = {'hostname': 'remote-gateway-07'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0598 - Composite inventory groups: core scenario

**Scenario 08/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [48]:
# Example 0598: Composite inventory groups - core
case = {'hostname': 'core-router-08'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0599 - Composite inventory groups: mgmt scenario

**Scenario 09/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [49]:
# Example 0599: Composite inventory groups - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


### Example 0600 - Composite inventory groups: backup scenario

**Scenario 10/10; pattern 060; Intermediate to advanced.** This is a self-contained offline exercise. Nested site inventory can be summarized without flattening every object manually.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Give a leaf count() returning one. Check the recursive result. Follow the class-specific steps under Pattern 060 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
devices: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This example assumes an acyclic tree with unique ownership; repeated references would count twice.

In [50]:
# Example 0600: Composite inventory groups - backup
case = {'hostname': 'backup-edge-10'}

class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def count(self):
        return 1

class Group:
    def __init__(self, children):
        self.children = tuple(children)

    def count(self):
        return sum(child.count() for child in self.children)

site = Group([Device(case["hostname"]), Group([Device("spare-a"), Device("spare-b")])])
assert site.count() == 3
print("devices:", site.count())


devices: 3


## Check your understanding: Observer, command, template, registry, and composite

**You should now be able to:** Represent events, reversible local commands, extension hooks, and recursive groups explicitly.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.